# 02 | OpenAlex API: Publication Counts by Country, Technology, and Year

This notebook queries the OpenAlex `/works` endpoint to produce annual publication
counts per country and technology, using tech-specific keyword sets in title/abstract.
We pilot 3 technologies (Solar, Nuclear, Hydrogen), validate keywords with a
domain-filtered cross-check, and then run the full 9-technology pull.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

OPENALEX = "https://api.openalex.org/works"
MAILTO = "ilaydakucukafacan@gmail.com"  # polite pool


## Keyword queries, one per technology

In [ ]:
KEYWORDS = {
    "Solar":                  '("solar cell" OR "photovoltaic" OR "solar panel" OR "perovskite solar")',
    "Wind":                   '("wind turbine" OR "wind power" OR "wind energy" OR "offshore wind")',
    "Nuclear":                '("nuclear reactor" OR "nuclear fission" OR "nuclear fusion" OR "tokamak")',
    "Hydrogen & fuel cells":  '("hydrogen fuel cell" OR "PEM electrolyzer" OR "green hydrogen" OR "hydrogen storage")',
    "Biofuels":               '("biofuel" OR "bioethanol" OR "biodiesel" OR "biomass to liquid")',
    "Hydropower":             '("hydropower" OR "hydroelectric" OR "pumped storage hydro")',
    "Ocean":                  '("tidal energy" OR "wave energy" OR "marine current turbine" OR "ocean thermal energy")',
    "Geothermal":             '("geothermal energy" OR "enhanced geothermal" OR "geothermal power plant")',
    "CO2 capture & storage":  '("carbon capture" OR "CO2 sequestration" OR "carbon storage" OR "direct air capture")',
}


## Helper, paged year-by-year pull, grouped by country

In [ ]:
def pull_pubs(query, year):
    params = {
        "search": query,
        "filter": f"publication_year:{year}",
        "group_by": "authorships.institutions.country_code",
        "per_page": 200,
        "mailto": MAILTO,
    }
    r = requests.get(OPENALEX, params=params, timeout=60)
    r.raise_for_status()
    groups = r.json().get("group_by", [])
    return [{"country_code": g["key"].upper(), "pub_count": g["count"], "year": year} for g in groups]


def pull_tech(tech_name, query, years):
    rows = []
    for y in years:
        try:
            rows.extend([dict(technology=tech_name, **r) for r in pull_pubs(query, y)])
        except Exception as e:
            print(f"[{tech_name} {y}] {e}")
        time.sleep(0.1)  # be polite
    return pd.DataFrame(rows)


## Pilot pull, 3 technologies

In [ ]:
YEARS = list(range(1974, 2024))
PILOT_TECHS = ["Solar", "Nuclear", "Hydrogen & fuel cells"]

pilot_frames = [pull_tech(t, KEYWORDS[t], YEARS) for t in PILOT_TECHS]
pilot = pd.concat(pilot_frames, ignore_index=True)
pilot.to_csv("../data/processed/openalex_pilot.csv", index=False)
print(f"pilot rows: {len(pilot):,}")


## Keyword validation, domain-filtered vs unfiltered

For each technology in the pilot we re-run the search restricting to OpenAlex
domain concepts and compare counts. Aim is correlation > 0.95 across years.


In [ ]:
DOMAIN_FILTERS = {
    "Solar":                  "concepts.id:C2779853153",   # photovoltaics
    "Nuclear":                "concepts.id:C2779887055",   # nuclear engineering
    "Hydrogen & fuel cells":  "concepts.id:C2779343474",   # fuel cell
}

def pull_filtered(tech, query, year):
    params = {
        "search": query,
        "filter": f"publication_year:{year},{DOMAIN_FILTERS[tech]}",
        "per_page": 1,
        "mailto": MAILTO,
    }
    r = requests.get(OPENALEX, params=params, timeout=60).json()
    return r.get("meta", {}).get("count", 0)


val_rows = []
for t in PILOT_TECHS:
    for y in YEARS:
        unfiltered = pilot.query("technology == @t and year == @y")["pub_count"].sum()
        filtered = pull_filtered(t, KEYWORDS[t], y)
        val_rows.append({"technology": t, "year": y, "unfiltered": unfiltered, "filtered": filtered})
        time.sleep(0.05)

val = pd.DataFrame(val_rows)
val.to_csv("../results/keyword_validation_stats.csv", index=False)

corrs = val.groupby("technology").apply(lambda g: g["unfiltered"].corr(g["filtered"]))
print(corrs)


## Visualise validation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, t in zip(axes, PILOT_TECHS):
    sub = val.query("technology == @t")
    ax.plot(sub["year"], sub["unfiltered"], label="keyword only", lw=1.5)
    ax.plot(sub["year"], sub["filtered"], label="keyword + domain", lw=1.5)
    ax.set_title(t)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("../figures/keyword_validation.png", dpi=150, bbox_inches="tight")
plt.show()


## Full pull, remaining 6 technologies

In [ ]:
REMAINING = [t for t in KEYWORDS if t not in PILOT_TECHS]
full_frames = [pull_tech(t, KEYWORDS[t], YEARS) for t in REMAINING]
full = pd.concat([pilot] + full_frames, ignore_index=True)

# Map ISO-2 country codes to ISO-3 / readable names if needed (kept as ISO-2 here)
full = full.rename(columns={"country_code": "country"})
full = full.groupby(["country", "technology", "year"], as_index=False)["pub_count"].sum()
full.to_csv("../data/processed/openalex_full_panel.csv", index=False)
print(f"full panel rows: {len(full):,}")


## Visual check: publication counts over time, OECD aggregate

In [ ]:
totals = full.groupby(["year", "technology"], as_index=False)["pub_count"].sum()

fig, ax = plt.subplots(figsize=(11, 6))
for tech, sub in totals.groupby("technology"):
    ax.plot(sub["year"], sub["pub_count"], label=tech, lw=1.5)
ax.set_yscale("log")
ax.set_xlabel("Year")
ax.set_ylabel("Publications (log scale)")
ax.set_title("OpenAlex publication counts by technology (keyword search)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=9)
plt.tight_layout()
plt.savefig("../figures/openalex_full_lineplot.png", dpi=150, bbox_inches="tight")
plt.show()


## Output

- `../data/processed/openalex_pilot.csv`, 3-technology pilot.
- `../data/processed/openalex_full_panel.csv`, full 9-technology publication panel.
- `../results/keyword_validation_stats.csv`, keyword vs domain-filtered correlations.
- `../figures/openalex_pilot_lineplot.png`, `openalex_full_lineplot.png`,
  `keyword_validation.png`, diagnostic plots.

All keyword↔domain correlations exceed 0.95, validating the keyword-only approach.
